# 23CSE301 Machine Learning - Capstone Project

## Classification Track - Part A

### Bank Marketing Dataset

**Algorithms:**
1. Logistic Regression
2. K-Nearest Neighbors
3. Gaussian Naive Bayes
4. Decision Tree Classifier
5. Support Vector Machine

---

### Problem Statement
The objective of this classification project is to predict whether a bank customer will subscribe to a term deposit (binary target variable `y`: `yes` or `no`) based on direct telemarketing campaign data from a Portuguese banking institution. The dataset encompasses client demographics, financial indicators, past contact interactions, and macroeconomic attributes. Accurately predicting customer subscription propensity enables the institution to optimize marketing resource allocation, target high-probability prospective clients, and improve campaign efficiency.

In accordance with the 23CSE301 Capstone Project Guidelines, this notebook implements Part A of the Classification Track, covering the five core algorithms evaluated in Review 1. All models are trained and tested on an identical stratified split with strict prevention of data leakage.


## 2. Import Required Libraries

This section imports the necessary libraries for data processing, exploratory data analysis, pipeline creation, model training, hyperparameter tuning, and performance evaluation.

A fixed random seed (`RANDOM_STATE = 42`) is established to guarantee reproducibility across all data splits and randomized estimators.


In [1]:
# Core data manipulation and numerical libraries
import os
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn model selection and validation
from sklearn.model_selection import train_test_split, GridSearchCV

# Scikit-learn preprocessing and pipeline utilities
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Scikit-learn classification algorithms (Part A)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC

# Scikit-learn evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

# Plot formatting and styling
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.autolayout"] = True

# Deterministic random seed
RANDOM_STATE = 42


## 3. Load Dataset

Loading the Bank Marketing dataset (`bank-full.csv`). The file uses semicolon delimiters (`sep=";"`). The path is verified to ensure robust loading whether executed from the project root or the notebooks directory.


In [2]:
# Locate bank-full.csv (checking current directory and common relative data paths)
csv_filename = "bank-full.csv"
if os.path.exists(csv_filename):
    csv_path = csv_filename
elif os.path.exists(os.path.join("data", csv_filename)):
    csv_path = os.path.join("data", csv_filename)
elif os.path.exists(os.path.join("..", "data", csv_filename)):
    csv_path = os.path.join("..", "data", csv_filename)
else:
    csv_path = csv_filename

df = pd.read_csv(csv_path, sep=";")
print(f"Dataset successfully loaded from: {csv_path}")


Dataset successfully loaded from: bank-full.csv


In [3]:
# Display first 5 rows
print("--- First 5 Rows ---")
display(df.head())

# Display last 5 rows
print("--- Last 5 Rows ---")
display(df.tail())

# Dataset shape
print(f"\nDataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")

# Column names and data types
print("\n--- Column Names & Data Types ---")
display(df.dtypes.to_frame(name="Data Type"))

# DataFrame info
print("\n--- DataFrame Information ---")
df.info()


--- First 5 Rows ---


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


--- Last 5 Rows ---


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
45206,51,technician,married,tertiary,no,825,no,no,cellular,17,nov,977,3,-1,0,unknown,yes
45207,71,retired,divorced,primary,no,1729,no,no,cellular,17,nov,456,2,-1,0,unknown,yes
45208,72,retired,married,secondary,no,5715,no,no,cellular,17,nov,1127,5,184,3,success,yes
45209,57,blue-collar,married,secondary,no,668,no,no,telephone,17,nov,508,4,-1,0,unknown,no
45210,37,entrepreneur,married,secondary,no,2971,no,no,cellular,17,nov,361,2,188,11,other,no



Dataset Dimensions: 45211 rows, 17 columns

--- Column Names & Data Types ---


,Data Type
age,int64
job,object
marital,object
education,object
default,object
balance,int64
housing,object
loan,object
contact,object
day,int64



--- DataFrame Information ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


## 4. Dataset Audit

A rigorous dataset audit assesses data hygiene prior to model construction:
- Verification of missing values (`NaN` / `null`)
- Detection of duplicate records
- Examination of numerical feature distributions and ranges
- Examination of categorical unique value counts and cardinality
- Identification and enumeration of implicit missing values recorded as `"unknown"`


In [4]:
# Missing values audit
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing_counts, "Percentage (%)": missing_pct})
print("--- Missing Values Audit ---")
display(missing_df)

# Duplicate records audit
duplicate_count = df.duplicated().sum()
print(f"\nDuplicate Rows Detected: {duplicate_count}")


--- Missing Values Audit ---


,Missing Count,Percentage (%)
age,0,0.0
job,0,0.0
marital,0,0.0
education,0,0.0
default,0,0.0
balance,0,0.0
housing,0,0.0
loan,0,0.0
contact,0,0.0
day,0,0.0



Duplicate Rows Detected: 0


In [5]:
# Statistical summary of numerical features
print("--- Numerical Features Statistical Summary ---")
display(df.describe().T)

# Statistical summary of categorical features
print("\n--- Categorical Features Statistical Summary ---")
display(df.describe(include=["object"]).T)


--- Numerical Features Statistical Summary ---


,count,mean,std,min,25%,50%,75%,max
age,45211.0,40.936210,10.618762,18.0,33.0,39.0,48.0,95.0
balance,45211.0,1362.272058,3044.765829,-8019.0,72.0,448.0,1428.0,102127.0
day,45211.0,15.806419,8.322476,1.0,8.0,16.0,21.0,31.0
duration,45211.0,258.163080,257.527812,0.0,103.0,180.0,319.0,4918.0
campaign,45211.0,2.763841,3.098021,1.0,1.0,2.0,3.0,63.0
pdays,45211.0,40.197828,100.128746,-1.0,-1.0,-1.0,-1.0,871.0
previous,45211.0,0.580323,2.303441,0.0,0.0,0.0,0.0,275.0



--- Categorical Features Statistical Summary ---


,count,unique,top,freq
job,45211,12,blue-collar,9732
marital,45211,3,married,27214
education,45211,4,secondary,23202
default,45211,2,no,44396
housing,45211,2,yes,25130
loan,45211,2,no,37967
contact,45211,3,cellular,29285
month,45211,12,may,13766
poutcome,45211,4,unknown,36959
y,45211,2,no,39922


In [6]:
# Categorical unique values audit
print("--- Categorical Unique Values ---")
cat_cols = df.select_dtypes(include=["object"]).columns
for col in cat_cols:
    print(f"Column '{col}' ({df[col].nunique()} unique): {df[col].unique().tolist()[:8]}")

# Audit 'unknown' values across columns
unknown_counts = (df == "unknown").sum()
unknown_df = pd.DataFrame({
    "Unknown Count": unknown_counts[unknown_counts > 0],
    "Percentage (%)": ((unknown_counts[unknown_counts > 0] / len(df)) * 100).round(2)
})
print("\n--- Implicit Missing Values ('unknown') Audit ---")
display(unknown_df)


--- Categorical Unique Values ---
Column 'job' (12 unique): ['management', 'technician', 'entrepreneur', 'blue-collar', 'unknown', 'retired', 'admin.', 'services']
Column 'marital' (3 unique): ['married', 'single', 'divorced']
Column 'education' (4 unique): ['tertiary', 'secondary', 'unknown', 'primary']
Column 'default' (2 unique): ['no', 'yes']
Column 'housing' (2 unique): ['yes', 'no']
Column 'loan' (2 unique): ['no', 'yes']
Column 'contact' (3 unique): ['unknown', 'cellular', 'telephone']
Column 'month' (12 unique): ['may', 'jun', 'jul', 'aug', 'oct', 'nov', 'dec', 'jan']
Column 'poutcome' (4 unique): ['unknown', 'failure', 'other', 'success']
Column 'y' (2 unique): ['no', 'yes']

--- Implicit Missing Values ('unknown') Audit ---


,Unknown Count,Percentage (%)
job,288,0.64
education,1857,4.11
contact,13020,28.80
poutcome,36959,81.75
